In [ ]:
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import os
from os.path import join
import string
import random
import re
import requests
import pandas as pd
from collections import Counter
from tqdm import tqdm

def tokenise( text ):
    tokens = []
    text = text.lower()
    text = re.sub( r'--' , ' -- ' , text)
    text = re.sub( r'-\"' , ' -- ' , text)
    text = re.sub( r'-\'' , ' -- ' , text)
    words = re.split( r'\s+' , text )
    for w in words:
        w = w.strip( string.punctuation )
        if re.search( r"[a-zA-Z]" , w ):
            tokens.append(w)
    return tokens

def remove_punctuation(words):
    new_list= []
    for w in words:
        if w.isalnum():
            new_list.append( w )
    return new_list


    
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.1 Safari/605.1.15',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 13_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.1 Safari/605.1.15'
]

def select_agent():
    n=random.randint(0,len(user_agents)-1)
    return user_agents[n]

def tag_visible(element):
    if element.parent.name in ['style', 'script', 'head', 'title', 'meta', '[document]']:
        return False
    if isinstance(element, Comment):
        return False
    return True

from io import StringIO
from html.parser import HTMLParser

class MLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.reset()
        self.strict = False
        self.convert_charrefs= True
        self.text = StringIO()
    def handle_data(self, d):
        self.text.write(d)
    def get_data(self):
        return self.text.getvalue()

def strip_tags(html):
    s = MLStripper()
    s.feed(html)
    text = s.get_data()
    text = re.sub( r'\s+' , ' ' , text )
    return text.strip()

## Add API key below
api_key = ''

out_dir = 'Goodreads_reviews'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)
    


# Download all reviews

In [ ]:
books_df = pd.read_excel('Books.xlsx')
all_ids = books_df['id'].unique().tolist()
all_ids = [gr_id for gr_id in all_ids if re.search('\d+',str(gr_id))]
print(len(all_ids))

out_dir = 'Goodreads_data'

def find_category(book_id):
    return books_df.query( f'id=={book_id}').iloc[0]['category']



In [ ]:
path = join(out_dir,f'user_engagement.csv')
out1 = open(path,'w',encoding='utf-8')
out1.write(f'id,title,rating_dist,ratings_count,text_reviews_count,average_rating,original_publication_year,category\n')


for book_id in tqdm(all_ids):

   
    #User engagement 
    api_call = 'https://www.goodreads.com/book/isbn/'
    api_call += f'?key={api_key}&id={book_id}&format=xml&text_only=true'
 
    #print(api_call)
    headers = {'User-Agent': select_agent() }
    response = requests.get( api_call,headers=headers)

    root = ET.fromstring(response.text)

    #cover_url = root.find('book/image_url').text
    rating_dist = root.find('book/work/rating_dist')
    if rating_dist is not None:
        rating_dist = rating_dist.text
    else:
        rating_dist = ''
    
    ratings_count = root.find('book/work/ratings_count')
    if ratings_count is not None:
        ratings_count = int(ratings_count.text)
    else:
        ratings_count = 0
        
    text_reviews_count = root.find('book/work/text_reviews_count')
    if text_reviews_count is not None:
        text_reviews_count = int(text_reviews_count.text)
    else:
        text_reviews_count = 0
        
    average_rating = root.find('book/average_rating')
    if average_rating is not None:
        average_rating = float(average_rating.text)
    else:
        average_rating = 0
        
    original_publication_year = root.find('book/work/original_publication_year')
    if original_publication_year is not None:
        original_publication_year = original_publication_year.text
    else:
        original_publication_year = None
    
    title = root.find( 'book/title' )
    if title is not None:
        title = title.text
        title = re.sub(r'["\']','',title)
    else:
        title = ''
        
    category = find_category(book_id)
    
    out1.write(f'{book_id},"{title}",{rating_dist},{ratings_count},{text_reviews_count},{average_rating},{original_publication_year},{category}\n')

out1.close()




## Download full reviews 

In [ ]:
for book_id in tqdm(all_ids):
    
    reviews_path = join(out_dir,f'reviews_{book_id}.tsv')
    
    if not os.path.exists(reviews_path):
        out2 = open(reviews_path,'w',encoding='utf-8')
        out2.write('review_id\ttype\tdate\tbody\tnr_comments\n')

        api_call = 'https://www.goodreads.com/book/isbn/'
        api_call += f'?key={api_key}&id={book_id}&format=xml&text_only=true'

        headers = {'User-Agent': select_agent() }
        response = requests.get( api_call,headers=headers)

        root = ET.fromstring(response.text)

        reviews_widget = root.find('book/reviews_widget').text
        reviews_widget =  '<body>' + reviews_widget + '</body>'

        soup = BeautifulSoup( reviews_widget ,"lxml")
        iframe = soup.find("iframe")

        pages_left = True
        url = iframe.get("src")

        i = 1
        while pages_left == True:
            if re.search( r'[&]page',url,re.IGNORECASE):
                url = url[:url.index('&page')]
            url += '&page=' + str(i)

            headers = {'User-Agent': select_agent() }
            response_reviews = requests.get(url,headers=headers)
            html_page = response_reviews.text

            if re.search(r'No\s+reviews\sfound',html_page,re.IGNORECASE):
                pages_left = False
            elif i == 100:
                pages_left = False


            soup = BeautifulSoup( html_page ,"lxml")

            review_links = soup.find_all('a',{'class':'gr_more_link'})
            for link in review_links:
                try: 
                    review_url = link.get('href')

                    review_id = os.path.basename(review_url)
                    index_questionmark = review_id.index('?')
                    review_id = review_id[:index_questionmark]

                    headers = {'User-Agent': select_agent() }

                    review_api = f"https://www.goodreads.com/review/show.xml?id={review_id}&key={api_key}"

                    headers = {'User-Agent': select_agent() }
                    response_review = requests.get(review_api,headers=headers)
                    review_page = response_review.text

                    tree = ET.fromstring(review_page)
                    review_body = tree.find('./review/body')
                    review_text = strip_tags(review_body.text)
                    review_date = '[Unknown]'

                    comments = tree.find('./review/comments')
                    if comments is not None:
                        nr_comments = len(comments)
                    else:
                        nr_comments = 0
                    out_line = f'{review_id}\toriginal\t{review_date}\t{review_text}\t{nr_comments}\n'
                    out2.write(out_line)
                    if comments is not None:
                        nr_comments = len(comments)
                        for comment in comments:
                            comment_body = comment.find('body')
                            comment_date = comment.find('created_at')

                            out_line = f'{review_id}\tcomment\t{comment_date.text}\t{comment_body.text}\t0\n'
                            out2.write(out_line)
                except:
                    print('An error occurred ...')

            i+=1
        out2.close()
